# Day 3 v3: ML Optimization — Vietnamese Price Prediction (<= 1M VND)

**Target:** RMSLE from 0.58 to <= 0.45

**Improvements:** Log-transform, Category features, Arch C (char_wb), Ridge, LightGBM Optuna, Blending

In [1]:
!nvidia-smi

Wed Apr 15 10:57:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   44C    P8              9W /  165W |     370MiB /  16380MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve().parent))

import random
import time
import pickle

import numpy as np
from tqdm.auto import tqdm
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

from pricer_vi.items import Item
from pricer_vi.evaluator import evaluate, rmsle

SEED = 42
DATASET = "SeanSunny/items_tv_v6"
PRICE_THRESHOLD = 1_000_000
CACHE_DIR = Path(".")
EVAL_SIZE = "all"
#EVAL_SIZE = "200"

random.seed(SEED)
np.random.seed(SEED)

## 1. Load Data + Filter

In [3]:
train, val, test = Item.from_hub(DATASET)
print(f"Raw: {len(train):,} train | {len(val):,} val | {len(test):,} test")

train = [item for item in train if item.price <= PRICE_THRESHOLD]
val = [item for item in val if item.price <= PRICE_THRESHOLD]
test = [item for item in test if item.price <= PRICE_THRESHOLD]
print(f"Filtered <= {PRICE_THRESHOLD:,} VND:")
print(f"  Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

train_prices = [item.price for item in train]
val_prices = [item.price for item in val]
prices = np.array(train_prices, dtype=float)
documents = [item.summary for item in train]
categories_train = [[item.category] for item in train]

log_prices = np.log1p(prices)
print(f"  Price: {prices.min():,.0f} - {prices.max():,.0f} VND | Mean: {prices.mean():,.0f} | Median: {np.median(prices):,.0f}")
print(f"  Log: {log_prices.min():.2f} - {log_prices.max():.2f} | Mean: {log_prices.mean():.2f}")

Raw: 110,000 train | 5,000 val | 5,000 test
Filtered <= 1,000,000 VND:
  Train: 85,727 | Val: 3,926 | Test: 3,872
  Price: 4,900 - 1,000,000 VND | Mean: 301,687 | Median: 229,000
  Log: 8.50 - 13.82 | Mean: 12.34


## 2. Tokenization (cache)

In [4]:
from underthesea import word_tokenize
from multiprocessing import Pool

def tokenize_one(text):
    return word_tokenize(text, format="text")

cache_train = CACHE_DIR / "tokenized_train_1m.pkl"
cache_test = CACHE_DIR / "tokenized_test_1m.pkl"
cache_val = CACHE_DIR / "tokenized_val_1m.pkl"

def load_or_tokenize(cache_path, texts, desc):
    if cache_path.exists():
        print(f"Loading: {cache_path}")
        with open(cache_path, "rb") as f:
            return pickle.load(f)
    t0 = time.time()
    print(f"Tokenizing {len(texts):,} docs...")
    with Pool(4) as p:
        result = list(tqdm(p.imap(tokenize_one, texts, chunksize=500), total=len(texts), desc=desc))
    print(f"  Done: {time.time()-t0:.1f}s")
    with open(cache_path, "wb") as f:
        pickle.dump(result, f)
    return result

tokenized_train = load_or_tokenize(cache_train, documents, "train")
tokenized_test = load_or_tokenize(cache_test, [item.summary for item in test], "test")
tokenized_val = load_or_tokenize(cache_val, [item.summary for item in val], "val")
tokenized_test_map = {item.summary: tok for item, tok in zip(test, tokenized_test)}

print(f"Sample: {tokenized_train[0][:100]}")

Loading: tokenized_train_1m.pkl
Loading: tokenized_test_1m.pkl
Loading: tokenized_val_1m.pkl
Sample: Tiêu_đề : Áo len_hoodie dày ấm cho nữ Danh_mục : Thời_trang nữ Thương_hiệu : LiLiLa Mô_tả : Hoodie c


## 3. Feature Engineering

In [5]:
# Arch B: word TF-IDF
t0 = time.time()
vectorizer_b = TfidfVectorizer(max_features=10000)
X_text_train = vectorizer_b.fit_transform(tokenized_train)
X_text_test = vectorizer_b.transform(tokenized_test)
X_text_val = vectorizer_b.transform(tokenized_val)
print(f"Arch B: {X_text_train.shape} ({time.time()-t0:.1f}s)")

# Arch C: word + char_wb
t0 = time.time()
arch_c = FeatureUnion([
    ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), max_features=5000)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=5000)),
])
X_text_c_train = arch_c.fit_transform(tokenized_train)
X_text_c_test = arch_c.transform(tokenized_test)
X_text_c_val = arch_c.transform(tokenized_val)
print(f"Arch C: {X_text_c_train.shape} ({time.time()-t0:.1f}s)")

# Category one-hot
cat_encoder = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_cat_train = cat_encoder.fit_transform(categories_train)
X_cat_test = cat_encoder.transform([[item.category] for item in test])
X_cat_val = cat_encoder.transform([[item.category] for item in val])
print(f"Categories: {list(cat_encoder.categories_[0])}")

# Combined
X_bc_train = hstack([X_text_train, X_cat_train])
X_bc_test = hstack([X_text_test, X_cat_test])
X_bc_val = hstack([X_text_val, X_cat_val])
X_cc_train = hstack([X_text_c_train, X_cat_train])
X_cc_test = hstack([X_text_c_test, X_cat_test])
X_cc_val = hstack([X_text_c_val, X_cat_val])
print(f"B+Cat: {X_bc_train.shape} | C+Cat: {X_cc_train.shape}")

Arch B: (85727, 10000) (2.4s)
Arch C: (85727, 10000) (29.2s)
Categories: ['Bách Hóa', 'Làm Đẹp - Sức Khỏe', 'Mẹ và Bé', 'Nhà Cửa - Đời Sống', 'Thời Trang', 'Ô Tô - Xe Máy', 'Điện Lạnh và Gia Dụng', 'Điện Tử - Công Nghệ']
B+Cat: (85727, 10008) | C+Cat: (85727, 10008)


In [6]:
# Helper: evaluate log-transform model
def evaluate_log_model(model, X_test_matrix, test_data, title):
    pred_log = model.predict(X_test_matrix)
    pred_price = np.clip(np.expm1(pred_log), 0, None)
    y_true = np.array([item.price for item in test_data], dtype=float)
    rmsle_val = rmsle(y_true, pred_price)
    mae_val = float(np.mean(np.abs(y_true - pred_price)))
    mask = y_true > 0
    mape_val = float(np.mean(np.abs((y_true[mask] - pred_price[mask]) / y_true[mask])) * 100)
    from sklearn.metrics import r2_score
    r2_val = r2_score(y_true, pred_price) * 100
    print(f"{title} ({len(test_data)} items):")
    print(f"  RMSLE: {rmsle_val:.4f} | MAE: {mae_val:,.0f} | MAPE: {mape_val:.1f}% | R2: {r2_val:.1f}%")
    return {"rmsle": rmsle_val, "mae": mae_val, "mape": mape_val, "r2": r2_val}

results = {}

---
## Phase 1: Log-transform — All Models

In [7]:
# Median baseline
training_median = float(np.median(prices))
def median_pricer(item):
    return training_median
results["Median (baseline)"] = evaluate(median_pricer, test, size=EVAL_SIZE)

  0%|          | 0/3872 [00:00<?, ?it/s]

250,400 130,000 203,000 131,000 103,000 456,000 55,000 112,000 179,000 81,000 40,000 16,000 298,000 177,000 95,000 40,000 99,000 80,000 119,000 6,000 204,500 721,000 6,000 64,000 129,000 70,000 33,000 114,000 586,000 110,000 58,000 150,000 161,000 40,000 153,160 140,000 89,000 60,000 665,000 606,000 66,000 5,500 220,999 91,000 139,001 551,000 179,000 64,000 143,000 80,000 165,750 136,000 141,000 750,000 154,000 10,000 417,558 24,750 96,000 79,000 596,000 9,000 116,000 144,000 29,000 4,000 30,000 287,465 351,000 154,000 6,000 110,000 40,000 221,000 40,000 110,000 321,000 30,000 120,000 72,000 16,000 461,000 21,000 461,000 180,000 178,700 80,000 130,000 120,000 130,000 320,000 30,000 76,000 661,000 130,000 40,000 146,000 571,000 150,000 89,000 19,000 50,000 1,500 70,000 334,000 7,000 21,000 190,500 39,000 9,000 113,700 18,000 770,000 59,000 226,000 169,000 150,000 620,000 125,000 114,000 212,000 87,000 0 133,000 347,000 42,170 60,000 130,000 625,000 59,000 201,000 181,000 270,000 174,000

In [8]:
# Ridge (B+Cat, log)
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_bc_train, log_prices)
results["Ridge (B+Cat, log)"] = evaluate_log_model(ridge_model, X_bc_test, test, "Ridge (B+Cat, log)")

Ridge (B+Cat, log) (3872 items):
  RMSLE: 0.5415 | MAE: 114,785 | MAPE: 46.6% | R2: 42.5%


In [9]:
# Ridge (C+Cat, log)
ridge_c = Ridge(alpha=1.0)
ridge_c.fit(X_cc_train, log_prices)
results["Ridge (C+Cat, log)"] = evaluate_log_model(ridge_c, X_cc_test, test, "Ridge (C+Cat, log)")

Ridge (C+Cat, log) (3872 items):
  RMSLE: 0.5697 | MAE: 121,307 | MAPE: 49.6% | R2: 37.3%


In [10]:
# LR (B+Cat, log) for comparison
lr_log = LinearRegression()
lr_log.fit(X_bc_train, log_prices)
results["LR (B+Cat, log)"] = evaluate_log_model(lr_log, X_bc_test, test, "LR (B+Cat, log)")

LR (B+Cat, log) (3872 items):
  RMSLE: 0.5493 | MAE: 117,714 | MAPE: 47.3% | R2: 36.3%


In [11]:
# Random Forest (B+Cat, log, subset 40K)
SUBSET_RF = 40_000
t0 = time.time()
rf_log = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=6, verbose=1)
rf_log.fit(X_bc_train[:SUBSET_RF], log_prices[:SUBSET_RF])
print(f"RF train: {time.time()-t0:.1f}s")
results["RF (B+Cat, log)"] = evaluate_log_model(rf_log, X_bc_test, test, "RF (B+Cat, log)")

[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:  2.9min
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed: 13.2min


RF train: 1255.7s
RF (B+Cat, log) (3872 items):
  RMSLE: 0.5801 | MAE: 124,061 | MAPE: 51.2% | R2: 34.5%


[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed: 20.9min finished
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed:    0.0s finished


In [12]:
# XGBoost (B+Cat, log)
t0 = time.time()
pbar_xgb = tqdm(total=1000, desc="XGBoost")
class XGBProgress(xgb.callback.TrainingCallback):
    def after_iteration(self, model, epoch, evals_log):
        pbar_xgb.update(1)
        return False
    def after_training(self, model):
        pbar_xgb.close()
        return model
xgb_log = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.1, random_state=SEED, n_jobs=6,
    tree_method="hist", callbacks=[XGBProgress()],
)
xgb_log.fit(X_bc_train, log_prices)
print(f"XGBoost train: {time.time()-t0:.1f}s")
results["XGBoost (B+Cat, log)"] = evaluate_log_model(xgb_log, X_bc_test, test, "XGBoost (B+Cat, log)")

XGBoost:   0%|          | 0/1000 [00:00<?, ?it/s]

XGBoost train: 202.6s
XGBoost (B+Cat, log) (3872 items):
  RMSLE: 0.5522 | MAE: 118,905 | MAPE: 48.1% | R2: 39.0%


In [13]:
# LightGBM (B+Cat, log)
t0 = time.time()
pbar_lgb = tqdm(total=1000, desc="LightGBM")
def lgb_cb(env): pbar_lgb.update(1)
lgb_log = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.1, random_state=SEED, n_jobs=6, verbose=-1)
lgb_log.fit(X_bc_train, log_prices, callbacks=[lgb_cb])
pbar_lgb.close()
print(f"LightGBM train: {time.time()-t0:.1f}s")
results["LightGBM (B+Cat, log)"] = evaluate_log_model(lgb_log, X_bc_test, test, "LightGBM (B+Cat, log)")

LightGBM:   0%|          | 0/1000 [00:00<?, ?it/s]

LightGBM train: 39.0s
LightGBM (B+Cat, log) (3872 items):
  RMSLE: 0.5286 | MAE: 112,889 | MAPE: 45.1% | R2: 44.7%


c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



In [14]:
# CatBoost (B+Cat, log)
t0 = time.time()
cb_log = CatBoostRegressor(iterations=1000, learning_rate=0.1, random_seed=SEED, verbose=100)
cb_log.fit(X_bc_train, log_prices)
print(f"CatBoost train: {time.time()-t0:.1f}s")
results["CatBoost (B+Cat, log)"] = evaluate_log_model(cb_log, X_bc_test, test, "CatBoost (B+Cat, log)")

0:	learn: 0.7621653	total: 340ms	remaining: 5m 40s
100:	learn: 0.6549305	total: 17.4s	remaining: 2m 35s
200:	learn: 0.6258439	total: 35.1s	remaining: 2m 19s
300:	learn: 0.6062159	total: 52.7s	remaining: 2m 2s
400:	learn: 0.5918573	total: 1m 9s	remaining: 1m 44s
500:	learn: 0.5803551	total: 1m 27s	remaining: 1m 26s
600:	learn: 0.5704149	total: 1m 44s	remaining: 1m 9s
700:	learn: 0.5614917	total: 2m 1s	remaining: 51.8s
800:	learn: 0.5536900	total: 2m 18s	remaining: 34.4s
900:	learn: 0.5465155	total: 2m 35s	remaining: 17.1s
999:	learn: 0.5399181	total: 2m 52s	remaining: 0us
CatBoost train: 173.5s
CatBoost (B+Cat, log) (3872 items):
  RMSLE: 0.5609 | MAE: 121,815 | MAPE: 49.4% | R2: 36.5%


In [15]:
# LightGBM (C+Cat, log)
t0 = time.time()
pbar_lgb2 = tqdm(total=1000, desc="LightGBM C")
def lgb_cb2(env): pbar_lgb2.update(1)
lgb_c_log = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.1, random_state=SEED, n_jobs=6, verbose=-1)
lgb_c_log.fit(X_cc_train, log_prices, callbacks=[lgb_cb2])
pbar_lgb2.close()
print(f"LightGBM C train: {time.time()-t0:.1f}s")
results["LightGBM (C+Cat, log)"] = evaluate_log_model(lgb_c_log, X_cc_test, test, "LightGBM (C+Cat, log)")

LightGBM C:   0%|          | 0/1000 [00:00<?, ?it/s]

LightGBM C train: 198.3s
LightGBM (C+Cat, log) (3872 items):
  RMSLE: 0.5392 | MAE: 114,505 | MAPE: 46.0% | R2: 43.0%


c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



---
## Phase 3: LightGBM Optuna Tuning

In [16]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 50, 200),
        "min_child_samples": trial.suggest_int("min_child_samples", 50, 200),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.3, 0.7),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.01, 1.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.01, 1.0, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.1, log=True),
        "n_estimators": 1500, "random_state": SEED, "n_jobs": 6, "verbose": -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_bc_train, log_prices)
    pred = np.clip(np.expm1(model.predict(X_bc_val)), 0, None)
    return rmsle(np.array(val_prices, dtype=float), pred)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50, show_progress_bar=True)
print(f"Best val RMSLE: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

  0%|          | 0/50 [00:00<?, ?it/s]

c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor

Best val RMSLE: 0.5609
Best params: {'num_leaves': 199, 'min_child_samples': 50, 'feature_fraction': 0.5449092780696718, 'lambda_l1': 0.02105329077470537, 'lambda_l2': 0.19038920070401877, 'learning_rate': 0.03833937236497283}


c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



In [17]:
# Train best on full train, eval on test
best_params = {**study.best_params, "n_estimators": 1500, "random_state": SEED, "n_jobs": 6, "verbose": -1}
lgb_tuned = lgb.LGBMRegressor(**best_params)
lgb_tuned.fit(X_bc_train, log_prices)
results["LightGBM Tuned (B+Cat, log)"] = evaluate_log_model(lgb_tuned, X_bc_test, test, "LightGBM Tuned (B+Cat, log)")

# Also Arch C
lgb_tuned_c = lgb.LGBMRegressor(**best_params)
lgb_tuned_c.fit(X_cc_train, log_prices)
results["LightGBM Tuned (C+Cat, log)"] = evaluate_log_model(lgb_tuned_c, X_cc_test, test, "LightGBM Tuned (C+Cat, log)")

c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



LightGBM Tuned (B+Cat, log) (3872 items):
  RMSLE: 0.5451 | MAE: 114,349 | MAPE: 46.9% | R2: 43.4%
LightGBM Tuned (C+Cat, log) (3872 items):
  RMSLE: 0.5255 | MAE: 110,879 | MAPE: 44.3% | R2: 45.8%


c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



---
## Phase 4: Weighted Blending

In [18]:
from scipy.optimize import minimize as scipy_minimize

pred_lgb_val = np.clip(np.expm1(lgb_tuned.predict(X_bc_val)), 0, None)
pred_xgb_val = np.clip(np.expm1(xgb_log.predict(X_bc_val)), 0, None)
pred_cb_val = np.clip(np.expm1(cb_log.predict(X_bc_val)), 0, None)
pred_lgb_c_val = np.clip(np.expm1(lgb_tuned_c.predict(X_cc_val)), 0, None)

val_true = np.array(val_prices, dtype=float)
val_preds = [pred_lgb_val, pred_xgb_val, pred_cb_val, pred_lgb_c_val]
names_blend = ["LGB Tuned B", "XGB B", "CB B", "LGB Tuned C"]

def blend_rmsle(w):
    w = np.abs(w); w = w / w.sum()
    return rmsle(val_true, sum(wi * p for wi, p in zip(w, val_preds)))

res_opt = scipy_minimize(blend_rmsle, x0=[0.4, 0.2, 0.2, 0.2], method="Nelder-Mead")
bw = np.abs(res_opt.x); bw = bw / bw.sum()
for n, w in zip(names_blend, bw): print(f"  {n}: {w:.3f}")
print(f"Val blend RMSLE: {res_opt.fun:.4f}")

# Apply to test
test_preds = [
    np.clip(np.expm1(lgb_tuned.predict(X_bc_test)), 0, None),
    np.clip(np.expm1(xgb_log.predict(X_bc_test)), 0, None),
    np.clip(np.expm1(cb_log.predict(X_bc_test)), 0, None),
    np.clip(np.expm1(lgb_tuned_c.predict(X_cc_test)), 0, None),
]
blended = sum(w * p for w, p in zip(bw, test_preds))
test_true = np.array([item.price for item in test], dtype=float)
from sklearn.metrics import r2_score
b_rmsle = rmsle(test_true, blended)
b_mae = float(np.mean(np.abs(test_true - blended)))
mask = test_true > 0
b_mape = float(np.mean(np.abs((test_true[mask] - blended[mask]) / test_true[mask])) * 100)
b_r2 = r2_score(test_true, blended) * 100
print(f"Blended Test: RMSLE={b_rmsle:.4f} | MAE={b_mae:,.0f} | MAPE={b_mape:.1f}% | R2={b_r2:.1f}%")
results["Blended (4 models)"] = {"rmsle": b_rmsle, "mae": b_mae, "mape": b_mape, "r2": b_r2}

c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



  LGB Tuned B: 0.163
  XGB B: 0.115
  CB B: 0.000
  LGB Tuned C: 0.721
Val blend RMSLE: 0.5283
Blended Test: RMSLE=0.5182 | MAE=109,705 | MAPE=44.2% | R2=47.0%


---
## Final Summary

In [19]:
import pandas as pd
summary = pd.DataFrame([{"Model": k, **v} for k, v in results.items()])
summary = summary.sort_values("rmsle")
print(summary.to_string(index=False))

best = min(results, key=lambda k: results[k]["rmsle"])
print(f"Best: {best} -- RMSLE={results[best]['rmsle']:.4f}")
print(f"v2 best: 0.5799 | v3 best: {results[best]['rmsle']:.4f} | Improvement: {(0.5799-results[best]['rmsle'])/0.5799*100:.1f}%")

                      Model    rmsle           mae      mape         r2
         Blended (4 models) 0.518237 109705.178966 44.152285  46.953615
LightGBM Tuned (C+Cat, log) 0.525491 110879.228926 44.343642  45.828806
      LightGBM (B+Cat, log) 0.528575 112889.258438 45.090809  44.679009
      LightGBM (C+Cat, log) 0.539171 114504.879933 45.980641  43.027769
         Ridge (B+Cat, log) 0.541478 114785.162224 46.642367  42.453605
LightGBM Tuned (B+Cat, log) 0.545105 114348.990793 46.873591  43.386418
            LR (B+Cat, log) 0.549264 117713.571898 47.290357  36.315735
       XGBoost (B+Cat, log) 0.552154 118904.675910 48.058620  39.014506
      CatBoost (B+Cat, log) 0.560946 121815.478201 49.426297  36.491199
         Ridge (C+Cat, log) 0.569693 121306.513786 49.624813  37.270673
            RF (B+Cat, log) 0.580133 124060.504676 51.200763  34.506164
          Median (baseline) 0.773557 169570.684659 77.409735 -10.360161
Best: Blended (4 models) -- RMSLE=0.5182
v2 best: 0.5799 | v3 be